# Notebook 26: Model Deployment – Joblib, Flask, FastAPI & Streamlit Basics
### Part 26/30 – ML Mastery Series for Python Experts

## From Notebook to Production – Deployment Reality Check

You built pipelines that are clean and safe — now let's get them out of the notebook and into the hands of real users. Time to serve predictions over the web… welcome to deployment!

- **Notebook ≠ production**: Interactive notebooks are for exploration; production requires robust, reproducible, stateless services
- **Need API/UI for inference**: Real users need simple interfaces — REST APIs for developers, web UIs for analysts, batch jobs for data pipelines
- **Serialization for consistency**: Saving the complete pipeline (preprocessing + model) ensures training and inference use identical transformations
- **Input validation & error handling**: Production systems crash gracefully; validate inputs, catch exceptions, return meaningful errors
- **Latency & throughput**: Prediction services must respond quickly (milliseconds) and handle concurrent requests efficiently
- **Logging & monitoring**: Track predictions, errors, and latency to detect drift, debug issues, and optimize performance
- **Scalability basics**: Stateless services can scale horizontally; understand workers, load balancing, and resource limits
- **Containerization preview**: Docker packages your entire environment (Python, dependencies, model) for consistent deployment anywhere

## Learning Objectives

By the end of this notebook, you will be able to:

- **Serialize & load full scikit-learn pipelines with joblib**: Save and restore complete preprocessing + model workflows
- **Create REST API with Flask & FastAPI**: Build HTTP endpoints that accept data and return predictions
- **Handle JSON input/output & validation**: Parse requests, validate schema, and return structured responses
- **Build interactive Streamlit app for model demo**: Create web UIs with sliders, buttons, and visualizations in pure Python
- **Add basic error handling & logging**: Graceful failure modes and operational visibility
- **Expose prediction endpoint**: Serve single and batch predictions over HTTP
- **Test API with Postman/cURL**: Verify endpoints work correctly with HTTP clients
- **Run Streamlit locally**: Launch and interact with your model through a browser
- **Prepare for cloud/container deployment**: Understand environment variables, port configuration, and Docker basics
- **Understand stateless prediction services**: Design services that don't retain user data between requests, enabling horizontal scaling

## 🚀 1. Training & Saving the Production Model

Before deployment, we need a trained model saved to disk. We'll use the California Housing dataset with a complete pipeline (ColumnTransformer + RandomForest) and serialize it with joblib.

In [1]:
import numpy as np
import pandas as pd
import joblib
import json
import logging
import warnings
warnings.filterwarnings('ignore')

# sklearn imports
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# For API examples (these would be in separate files in production)
# from flask import Flask, request, jsonify
# from fastapi import FastAPI, HTTPException
# from pydantic import BaseModel
# import streamlit as st

print("Libraries loaded successfully")
print("Note: Flask, FastAPI, and Streamlit are shown as code templates")
print("Run them in separate .py files for actual serving")

Libraries loaded successfully
Note: Flask, FastAPI, and Streamlit are shown as code templates
Run them in separate .py files for actual serving


In [2]:
# Load California Housing dataset
housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = housing.target

print(f"Dataset: California Housing")
print(f"Samples: {X.shape[0]}, Features: {X.shape[1]}")
print(f"Target: Median house value (in $100,000s)")
print(f"\nFeature names: {list(X.columns)}")
print(f"\nFirst 3 rows:")
print(X.head(3))

Dataset: California Housing
Samples: 20640, Features: 8
Target: Median house value (in $100,000s)

Feature names: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']

First 3 rows:
   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   

   Longitude  
0    -122.23  
1    -122.22  
2    -122.24  


In [3]:
# Add synthetic categorical feature for demonstration
# In real deployment, this would come from actual categorical data
X['ocean_proximity'] = pd.cut(
    X['Latitude'],
    bins=[32, 35, 38, 41, 44],
    labels=['NEAR_BAY', 'INLAND', '<1H_OCEAN', 'NEAR_OCEAN']
)

# Introduce some missing values (realistic scenario)
np.random.seed(42)
missing_idx = np.random.choice(X.index, size=100, replace=False)
X.loc[missing_idx[:50], 'AveRooms'] = np.nan
X.loc[missing_idx[50:], 'AveBedrms'] = np.nan

# Define column types
numeric_features = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
categorical_features = ['ocean_proximity']

print(f"Numeric features ({len(numeric_features)}): {numeric_features[:3]}...")
print(f"Categorical features ({len(categorical_features)}): {categorical_features}")
print(f"\nMissing values: \n{X.isnull().sum()}")

Numeric features (8): ['MedInc', 'HouseAge', 'AveRooms']...
Categorical features (1): ['ocean_proximity']

Missing values: 
MedInc              0
HouseAge            0
AveRooms           50
AveBedrms          50
Population          0
AveOccup            0
Latitude            0
Longitude           0
ocean_proximity     0
dtype: int64


In [5]:
# Build production-ready pipeline
# Numeric preprocessing: impute missing values, then scale
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical preprocessing: one-hot encode
categorical_transformer = Pipeline([
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    # handle_unknown='ignore' prevents crashes on unseen categories
])

# Combine preprocessing
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# Full pipeline with RandomForest
production_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])

# Split and train (in production, you'd use full data or scheduled retraining)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
production_pipe.fit(X_train, y_train)

# Verify performance
y_pred = production_pipe.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Model trained. Test RMSE: {rmse:.4f}")

# Show pipeline structure
print(f"\nPipeline steps:")
for name, step in production_pipe.named_steps.items():
    print(f"  {name}: {step.__class__.__name__}")

Model trained. Test RMSE: 0.5054

Pipeline steps:
  preprocessor: ColumnTransformer
  regressor: RandomForestRegressor


In [6]:
# Serialize the complete pipeline to disk
model_filename = 'housing_model_v1.joblib'
joblib.dump(production_pipe, model_filename, compress=3)

# Verify file was created and check size
import os
file_size = os.path.getsize(model_filename) / (1024 * 1024)  # MB
print(f"Model saved to: {model_filename}")
print(f"File size: {file_size:.2f} MB")
print(f"Compression level: 3 (balanced size/speed)")

# Load back to verify serialization worked
loaded_model = joblib.load(model_filename)
test_pred = loaded_model.predict(X_test.iloc[:5])
print(f"\nVerification: Loaded model predicts same values")
print(f"Original predictions: {production_pipe.predict(X_test.iloc[:5])}")
print(f"Loaded predictions:   {test_pred}")
print(f"Match: {np.allclose(production_pipe.predict(X_test.iloc[:5]), test_pred)}")

# Clean up (in production, keep the file)
# os.remove(model_filename)
print(f"\nModel ready for deployment at: {model_filename}")

Model saved to: housing_model_v1.joblib
File size: 31.75 MB
Compression level: 3 (balanced size/speed)

Verification: Loaded model predicts same values
Original predictions: [0.54259   0.75142   4.8984272 2.33117   2.26524  ]
Loaded predictions:   [0.54259   0.75142   4.8984272 2.33117   2.26524  ]
Match: True

Model ready for deployment at: housing_model_v1.joblib


## 🌶️ 2. Basic Flask API – Simple Prediction Endpoint

Flask is a lightweight WSGI web framework perfect for simple ML APIs. We'll create a `/predict` endpoint that loads our model, accepts JSON input, and returns predictions.

**Note**: Run this code in a separate `app.py` file, not in Jupyter.

In [7]:
# Flask API Template (save as flask_app.py)
flask_code = '''
# flask_app.py - Flask API for Housing Price Prediction
from flask import Flask, request, jsonify
import joblib
import numpy as np
import pandas as pd
import logging
from datetime import datetime

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('api.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# Initialize Flask app
app = Flask(__name__)

# Load model at startup (not per request - important for performance)
MODEL_PATH = 'housing_model_v1.joblib'
model = joblib.load(MODEL_PATH)
logger.info(f"Model loaded from {MODEL_PATH}")

@app.route('/health', methods=['GET'])
def health():
    """Health check endpoint."""
    return jsonify({'status': 'healthy', 'model': 'housing_v1', 'timestamp': datetime.now().isoformat()})

@app.route('/predict', methods=['POST'])
def predict():
    """
    Prediction endpoint.
    Expects JSON with feature values.
    Returns predicted house price.
    """
    try:
        # Get JSON data from request
        data = request.get_json(force=True)
        logger.info(f"Received prediction request: {data}")
        
        # Convert to DataFrame (handles both single and batch)
        if isinstance(data, dict):
            # Single prediction
            df = pd.DataFrame([data])
        elif isinstance(data, list):
            # Batch prediction
            df = pd.DataFrame(data)
        else:
            return jsonify({'error': 'Invalid input format. Expected dict or list.'}), 400
        
        # Validate required features
        required_features = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 
                           'Population', 'AveOccup', 'Latitude', 'Longitude', 'ocean_proximity']
        missing = [f for f in required_features if f not in df.columns]
        if missing:
            return jsonify({'error': f'Missing features: {missing}'}), 400
        
        # Make prediction
        predictions = model.predict(df)
        
        # Prepare response
        response = {
            'predictions': predictions.tolist(),
            'model_version': 'v1',
            'timestamp': datetime.now().isoformat()
        }
        
        logger.info(f"Prediction successful: {response['predictions']}")
        return jsonify(response)
        
    except Exception as e:
        logger.error(f"Prediction error: {str(e)}")
        return jsonify({'error': str(e)}), 500

if __name__ == '__main__':
    # Run development server (use gunicorn in production)
    app.run(host='0.0.0.0', port=5000, debug=True)
'''

# Write to file
with open('flask_app.py', 'w') as f:
    f.write(flask_code)

print("Flask app template written to flask_app.py")
print("\nTo run:")
print("  1. Ensure housing_model_v1.joblib is in the same directory")
print("  2. Run: python flask_app.py")
print("  3. Test with cURL (see below)")

Flask app template written to flask_app.py

To run:
  1. Ensure housing_model_v1.joblib is in the same directory
  2. Run: python flask_app.py
  3. Test with cURL (see below)


In [8]:
# Example cURL commands to test the Flask API
curl_examples = '''
# Test health endpoint
curl http://localhost:5000/health

# Single prediction
curl -X POST http://localhost:5000/predict \\
  -H "Content-Type: application/json" \\
  -d '{
    "MedInc": 8.3252,
    "HouseAge": 41.0,
    "AveRooms": 6.984,
    "AveBedrms": 1.023,
    "Population": 322.0,
    "AveOccup": 2.555,
    "Latitude": 37.88,
    "Longitude": -122.23,
    "ocean_proximity": "NEAR_BAY"
  }'

# Batch prediction
curl -X POST http://localhost:5000/predict \\
  -H "Content-Type: application/json" \\
  -d '[
    {"MedInc": 8.3252, "HouseAge": 41.0, "AveRooms": 6.984, "AveBedrms": 1.023, 
     "Population": 322.0, "AveOccup": 2.555, "Latitude": 37.88, "Longitude": -122.23, 
     "ocean_proximity": "NEAR_BAY"},
    {"MedInc": 3.5, "HouseAge": 20.0, "AveRooms": 5.5, "AveBedrms": 1.1, 
     "Population": 800.0, "AveOccup": 3.0, "Latitude": 34.0, "Longitude": -118.0, 
     "ocean_proximity": "INLAND"}
  ]'
'''

print("cURL examples for testing Flask API:")
print(curl_examples)

# Simulate what the API would return
test_input = pd.DataFrame([{
    'MedInc': 8.3252, 'HouseAge': 41.0, 'AveRooms': 6.984, 'AveBedrms': 1.023,
    'Population': 322.0, 'AveOccup': 2.555, 'Latitude': 37.88, 'Longitude': -122.23,
    'ocean_proximity': 'NEAR_BAY'
}])
test_pred = loaded_model.predict(test_input)[0]
print(f"\nSimulated API response for test input:")
print(f"  Predicted value: ${test_pred * 100000:.2f}")
print(f"  (Model outputs in $100,000s)")

cURL examples for testing Flask API:

# Test health endpoint
curl http://localhost:5000/health

# Single prediction
curl -X POST http://localhost:5000/predict \
  -H "Content-Type: application/json" \
  -d '{
    "MedInc": 8.3252,
    "HouseAge": 41.0,
    "AveRooms": 6.984,
    "AveBedrms": 1.023,
    "Population": 322.0,
    "AveOccup": 2.555,
    "Latitude": 37.88,
    "Longitude": -122.23,
    "ocean_proximity": "NEAR_BAY"
  }'

# Batch prediction
curl -X POST http://localhost:5000/predict \
  -H "Content-Type: application/json" \
  -d '[
    {"MedInc": 8.3252, "HouseAge": 41.0, "AveRooms": 6.984, "AveBedrms": 1.023, 
     "Population": 322.0, "AveOccup": 2.555, "Latitude": 37.88, "Longitude": -122.23, 
     "ocean_proximity": "NEAR_BAY"},
    {"MedInc": 3.5, "HouseAge": 20.0, "AveRooms": 5.5, "AveBedrms": 1.1, 
     "Population": 800.0, "AveOccup": 3.0, "Latitude": 34.0, "Longitude": -118.0, 
     "ocean_proximity": "INLAND"}
  ]'


Simulated API response for test input:
  Predict

## ⚡ 3. FastAPI – Modern, Type-Safe Alternative

FastAPI provides automatic validation, serialization, and documentation (OpenAPI/Swagger) based on Python type hints. It's significantly faster than Flask due to async support and pydantic validation.

In [9]:
# FastAPI Template (save as fastapi_app.py)
fastapi_code = '''
# fastapi_app.py - FastAPI for Housing Price Prediction
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field, validator
import joblib
import numpy as np
import pandas as pd
import logging
from datetime import datetime
from typing import List, Union
import uvicorn

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Initialize FastAPI app
app = FastAPI(
    title="Housing Price Prediction API",
    description="Predict California housing prices using ML",
    version="1.0.0"
)

# Load model at startup
MODEL_PATH = 'housing_model_v1.joblib'
model = joblib.load(MODEL_PATH)
logger.info(f"Model loaded from {MODEL_PATH}")

# Pydantic model for input validation
class HouseFeatures(BaseModel):
    """Input schema for house features with validation."""
    MedInc: float = Field(..., gt=0, lt=20, description="Median income in block group")
    HouseAge: float = Field(..., gt=0, lt=100, description="Median house age")
    AveRooms: float = Field(..., gt=0, lt=100, description="Average rooms per household")
    AveBedrms: float = Field(..., gt=0, lt=50, description="Average bedrooms per household")
    Population: float = Field(..., gt=0, lt=50000, description="Block group population")
    AveOccup: float = Field(..., gt=0, lt=100, description="Average household occupancy")
    Latitude: float = Field(..., gt=32, lt=43, description="Latitude")
    Longitude: float = Field(..., lt=-114, gt=-125, description="Longitude")
    ocean_proximity: str = Field(..., description="Ocean proximity category")
    
    @validator('ocean_proximity')
    def validate_ocean(cls, v):
        allowed = ['NEAR_BAY', 'INLAND', '<1H_OCEAN', 'NEAR_OCEAN']
        if v not in allowed:
            raise ValueError(f'ocean_proximity must be one of {allowed}')
        return v

class PredictionResponse(BaseModel):
    """Output schema for predictions."""
    predictions: List[float]
    model_version: str
    timestamp: str

@app.get("/health")
async def health():
    """Health check endpoint."""
    return {
        "status": "healthy",
        "model": "housing_v1",
        "timestamp": datetime.now().isoformat()
    }

@app.post("/predict", response_model=PredictionResponse)
async def predict(features: Union[HouseFeatures, List[HouseFeatures]]):
    """
    Predict house prices.
    Accepts single house or list of houses.
    """
    try:
        # Convert to DataFrame
        if isinstance(features, list):
            df = pd.DataFrame([f.dict() for f in features])
            logger.info(f"Batch prediction for {len(features)} houses")
        else:
            df = pd.DataFrame([features.dict()])
            logger.info(f"Single prediction for house")
        
        # Predict
        predictions = model.predict(df)
        
        return PredictionResponse(
            predictions=predictions.tolist(),
            model_version="v1",
            timestamp=datetime.now().isoformat()
        )
        
    except Exception as e:
        logger.error(f"Prediction error: {str(e)}")
        raise HTTPException(status_code=500, detail=str(e))

@app.get("/docs", include_in_schema=False)
async def custom_docs():
    """Redirect to Swagger UI."""
    return {"message": "Visit /docs for Swagger UI or /redoc for ReDoc"}

if __name__ == "__main__":
    # Run with uvicorn
    uvicorn.run(app, host="0.0.0.0", port=8000)
'''

with open('fastapi_app.py', 'w') as f:
    f.write(fastapi_code)

print("FastAPI app template written to fastapi_app.py")
print("\nTo run:")
print("  1. Ensure housing_model_v1.joblib is in the same directory")
print("  2. Run: uvicorn fastapi_app:app --reload")
print("  3. Visit http://localhost:8000/docs for interactive Swagger UI")
print("  4. Test with cURL or the web interface")

FastAPI app template written to fastapi_app.py

To run:
  1. Ensure housing_model_v1.joblib is in the same directory
  2. Run: uvicorn fastapi_app:app --reload
  3. Visit http://localhost:8000/docs for interactive Swagger UI
  4. Test with cURL or the web interface


In [10]:
# FastAPI test examples and benefits
fastapi_benefits = """
FastAPI Advantages over Flask:
1. Automatic validation via Pydantic (catches bad input before reaching model)
2. Type hints generate OpenAPI docs automatically (/docs endpoint)
3. Async support for better concurrency (async/await)
4. Native JSON schema validation with meaningful error messages
5. Faster performance (Starlette + Pydantic)
6. Dependency injection system for clean code

Test commands:
# Health check
curl http://localhost:8000/health

# Valid prediction
curl -X POST "http://localhost:8000/predict" \\
  -H "Content-Type: application/json" \\
  -d '{
    "MedInc": 8.3252,
    "HouseAge": 41.0,
    "AveRooms": 6.984,
    "AveBedrms": 1.023,
    "Population": 322.0,
    "AveOccup": 2.555,
    "Latitude": 37.88,
    "Longitude": -122.23,
    "ocean_proximity": "NEAR_BAY"
  }'

# Invalid prediction (will return 422 validation error)
curl -X POST "http://localhost:8000/predict" \\
  -H "Content-Type: application/json" \\
  -d '{
    "MedInc": -5,
    "HouseAge": 41.0,
    "AveRooms": 6.984,
    "AveBedrms": 1.023,
    "Population": 322.0,
    "AveOccup": 2.555,
    "Latitude": 37.88,
    "Longitude": -122.23,
    "ocean_proximity": "INVALID"
  }'
"""

print(fastapi_benefits)


FastAPI Advantages over Flask:
1. Automatic validation via Pydantic (catches bad input before reaching model)
2. Type hints generate OpenAPI docs automatically (/docs endpoint)
3. Async support for better concurrency (async/await)
4. Native JSON schema validation with meaningful error messages
5. Faster performance (Starlette + Pydantic)
6. Dependency injection system for clean code

Test commands:
# Health check
curl http://localhost:8000/health

# Valid prediction
curl -X POST "http://localhost:8000/predict" \
  -H "Content-Type: application/json" \
  -d '{
    "MedInc": 8.3252,
    "HouseAge": 41.0,
    "AveRooms": 6.984,
    "AveBedrms": 1.023,
    "Population": 322.0,
    "AveOccup": 2.555,
    "Latitude": 37.88,
    "Longitude": -122.23,
    "ocean_proximity": "NEAR_BAY"
  }'

# Invalid prediction (will return 422 validation error)
curl -X POST "http://localhost:8000/predict" \
  -H "Content-Type: application/json" \
  -d '{
    "MedInc": -5,
    "HouseAge": 41.0,
    "AveRooms"

## 🎈 4. Streamlit – Interactive Web UI in Minutes

Streamlit turns data scripts into shareable web apps in minutes. Perfect for ML demos, internal tools, and rapid prototyping. No HTML/CSS/JS required — pure Python.

In [11]:
# Streamlit App Template (save as streamlit_app.py)
streamlit_code = '''
# streamlit_app.py - Interactive Housing Price Predictor
import streamlit as st
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Page configuration
st.set_page_config(
    page_title="California Housing Price Predictor",
    page_icon="🏠",
    layout="wide"
)

# Load model (cached to avoid reloading)
@st.cache_resource
def load_model():
    return joblib.load('housing_model_v1.joblib')

model = load_model()

# Title and description
st.title("🏠 California Housing Price Predictor")
st.markdown("""
Enter house features to get a price prediction.
This model was trained on California census block data.
""")

# Create two columns for inputs
col1, col2 = st.columns(2)

with col1:
    st.subheader("Economic & Demographics")
    med_income = st.slider("Median Income ($10k)", 0.5, 15.0, 3.0, 0.1)
    house_age = st.slider("House Age (years)", 1, 52, 20, 1)
    population = st.number_input("Population", 100, 35000, 1000, 100)

with col2:
    st.subheader("Property & Location")
    ave_rooms = st.slider("Avg Rooms", 1.0, 10.0, 5.0, 0.1)
    ave_bedrms = st.slider("Avg Bedrooms", 0.5, 5.0, 1.0, 0.1)
    ave_occup = st.slider("Avg Occupancy", 1.0, 10.0, 3.0, 0.1)
    
    location = st.selectbox(
        "Ocean Proximity",
        ['NEAR_BAY', 'INLAND', '<1H_OCEAN', 'NEAR_OCEAN']
    )

# Location coordinates (simplified)
lat_lon_map = {
    'NEAR_BAY': (37.8, -122.2),
    'INLAND': (37.0, -120.0),
    '<1H_OCEAN': (34.0, -118.0),
    'NEAR_OCEAN': (36.0, -122.0)
}
latitude, longitude = lat_lon_map[location]

# Prediction button
if st.button("🔮 Predict Price", type="primary"):
    # Create input DataFrame
    input_data = pd.DataFrame([{
        'MedInc': med_income,
        'HouseAge': house_age,
        'AveRooms': ave_rooms,
        'AveBedrms': ave_bedrms,
        'Population': population,
        'AveOccup': ave_occup,
        'Latitude': latitude,
        'Longitude': longitude,
        'ocean_proximity': location
    }])
    
    # Predict
    prediction = model.predict(input_data)[0]
    price_usd = prediction * 100000
    
    # Display result
    st.success(f"### Predicted Price: ${price_usd:,.0f}")
    
    # Progress bar showing relative value
    st.progress(min(prediction / 5.0, 1.0))
    st.caption(f"Relative value index: {prediction:.2f}/5.0")
    
    # Feature importance (if available)
    if hasattr(model.named_steps['regressor'], 'feature_importances_'):
        st.subheader("Feature Importance")
        importances = model.named_steps['regressor'].feature_importances_
        # Simplified - in reality need to map to original features post-preprocessing
        st.bar_chart({
            'Income': importances[0] if len(importances) > 0 else 0.3,
            'Location': importances[1] if len(importances) > 1 else 0.2,
            'Age': importances[2] if len(importances) > 2 else 0.15,
            'Rooms': importances[3] if len(importances) > 3 else 0.15,
            'Other': 0.2
        })

# Sidebar info
st.sidebar.header("About")
st.sidebar.info("""
This app demonstrates deploying a scikit-learn model 
with Streamlit. The model predicts median house values 
in California districts based on 1990 census data.
""")

st.sidebar.header("Model Info")
st.sidebar.json({
    "type": "RandomForestRegressor",
    "features": 9,
    "target": "Median House Value ($100k)",
    "version": "1.0"
})
'''

with open('streamlit_app.py', 'w') as f:
    f.write(streamlit_code)

print("Streamlit app template written to streamlit_app.py")
print("\nTo run:")
print("  streamlit run streamlit_app.py")
print("\nFeatures:")
print("  - Interactive sliders and inputs")
print("  - Real-time prediction on button click")
print("  - Progress bar and visual feedback")
print("  - Responsive layout (works on mobile)")
print("  - Automatic caching of model loading")

Streamlit app template written to streamlit_app.py

To run:
  streamlit run streamlit_app.py

Features:
  - Interactive sliders and inputs
  - Real-time prediction on button click
  - Progress bar and visual feedback
  - Responsive layout (works on mobile)
  - Automatic caching of model loading


## 🛡️ 5. Input Validation, Error Handling & Logging

Production systems must handle bad gracefully. Pydantic provides declarative validation, while logging gives operational visibility.

In [13]:
# Demonstrate validation concepts (would be in FastAPI app)
from pydantic import BaseModel, Field, validator, ValidationError

class ValidatedInput(BaseModel):
    """Example of comprehensive input validation."""
    MedInc: float = Field(..., gt=0, lt=20, description="Median income")
    HouseAge: float = Field(..., ge=1, le=100, description="House age")
    
    @validator('MedInc')
    def income_reasonable(cls, v):
        if v > 15:
            raise ValueError('Income seems unusually high, please verify')
        return v
    
    @validator('HouseAge')
    def age_positive(cls, v):
        if v < 0:
            raise ValueError('Age cannot be negative')
        return v

# Test validation
try:
    valid = ValidatedInput(MedInc=8.5, HouseAge=25)
    print(f"✓ Valid input accepted: {valid.dict()}")
except ValidationError as e:
    print(f"✗ Validation error: {e}")

# Test invalid input
try:
    invalid = ValidatedInput(MedInc=-5, HouseAge=25)
except ValidationError as e:
    print(f"✗ Invalid input rejected (as expected):")
    print(f"  Error: {e.errors()[0]['msg']}")

# Test edge case
try:
    edge = ValidatedInput(MedInc=18, HouseAge=150)  # Income OK, age too high
except ValidationError as e:
    print(f"✗ Edge case rejected:")
    print(f"  Error: {e.errors()[0]['msg']}")

✓ Valid input accepted: {'MedInc': 8.5, 'HouseAge': 25.0}
✗ Invalid input rejected (as expected):
  Error: Input should be greater than 0
✗ Edge case rejected:
  Error: Value error, Income seems unusually high, please verify


In [14]:
# Logging setup for production
import logging
from datetime import datetime

# Configure structured logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(name)s | %(message)s',
    handlers=[
        logging.FileHandler('model_api.log'),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger('housing_api')

# Simulate prediction logging
def log_prediction(input_data, prediction, duration_ms, error=None):
    """Structured logging for predictions."""
    log_entry = {
        'timestamp': datetime.now().isoformat(),
        'event': 'prediction',
        'input_features': {k: float(v) if isinstance(v, (int, float)) else str(v) 
                          for k, v in input_data.items()},
        'prediction': float(prediction) if prediction else None,
        'latency_ms': duration_ms,
        'error': str(error) if error else None,
        'model_version': 'v1'
    }
    
    if error:
        logger.error(f"Prediction failed: {log_entry}")
    else:
        logger.info(f"Prediction success: {log_entry}")
    
    return log_entry

# Simulate successful prediction
test_input = {'MedInc': 8.5, 'HouseAge': 25, 'AveRooms': 6.0}
log_prediction(test_input, 2.5, 45)

# Simulate failed prediction
log_prediction(test_input, None, 120, error="Missing feature: ocean_proximity")

print("\nLogging examples shown above")
print("In production, these would be written to file and/or sent to log aggregation service")

# Read log file
print("\nLog file contents:")
try:
    with open('model_api.log', 'r') as f:
        lines = f.readlines()[-3:]  # Last 3 lines
        for line in lines:
            print(line.strip())
except:
    print("(Log file not yet created)")

2026-03-21 10:09:26,507 | INFO | housing_api | Prediction success: {'timestamp': '2026-03-21T10:09:26.507367', 'event': 'prediction', 'input_features': {'MedInc': 8.5, 'HouseAge': 25.0, 'AveRooms': 6.0}, 'prediction': 2.5, 'latency_ms': 45, 'error': None, 'model_version': 'v1'}
2026-03-21 10:09:26,510 | ERROR | housing_api | Prediction failed: {'timestamp': '2026-03-21T10:09:26.510839', 'event': 'prediction', 'input_features': {'MedInc': 8.5, 'HouseAge': 25.0, 'AveRooms': 6.0}, 'prediction': None, 'latency_ms': 120, 'error': 'Missing feature: ocean_proximity', 'model_version': 'v1'}



Logging examples shown above
In production, these would be written to file and/or sent to log aggregation service

Log file contents:
2026-03-21 10:09:26,507 | INFO | housing_api | Prediction success: {'timestamp': '2026-03-21T10:09:26.507367', 'event': 'prediction', 'input_features': {'MedInc': 8.5, 'HouseAge': 25.0, 'AveRooms': 6.0}, 'prediction': 2.5, 'latency_ms': 45, 'error': None, 'model_version': 'v1'}
2026-03-21 10:09:26,510 | ERROR | housing_api | Prediction failed: {'timestamp': '2026-03-21T10:09:26.510839', 'event': 'prediction', 'input_features': {'MedInc': 8.5, 'HouseAge': 25.0, 'AveRooms': 6.0}, 'prediction': None, 'latency_ms': 120, 'error': 'Missing feature: ocean_proximity', 'model_version': 'v1'}


## 🧪 6. Testing the Deployed Model

Testing ensures your API works correctly before production. We'll demonstrate batch testing and compare API predictions to notebook results.

In [19]:
# Simulate API client testing
import requests  # Would use this in real testing
from time import time

def simulate_api_predict(data, model=loaded_model):
    """
    Simulate API prediction (in real scenario, this would be HTTP request).
    Returns prediction and simulated latency.
    """
    import time as _time
    start = _time.time()
    
    # Simulate network latency (20-50ms)
    import random
    latency = random.uniform(0.02, 0.05)
    
    # Convert to DataFrame and predict
    if isinstance(data, dict):
        df = pd.DataFrame([data])
    else:
        df = pd.DataFrame(data)
    
    prediction = model.predict(df)
    
    # Simulate processing time
    processing_time = _time.time() - start
    total_time = processing_time + latency
    
    return {
        'predictions': prediction.tolist(),
        'latency_ms': total_time * 1000,
        'status': 'success'
    }

# Test with sample data
test_samples = [
    {'MedInc': 8.3252, 'HouseAge': 41.0, 'AveRooms': 6.984, 'AveBedrms': 1.023,
     'Population': 322.0, 'AveOccup': 2.555, 'Latitude': 37.88, 'Longitude': -122.23,
     'ocean_proximity': 'NEAR_BAY'},
    {'MedInc': 3.5, 'HouseAge': 20.0, 'AveRooms': 5.5, 'AveBedrms': 1.1,
     'Population': 800.0, 'AveOccup': 3.0, 'Latitude': 34.0, 'Longitude': -118.0,
     'ocean_proximity': 'INLAND'}
]

print("Simulated API Testing:")
print("="*60)
for i, sample in enumerate(test_samples, 1):
    result = simulate_api_predict(sample)
    print(f"\nTest {i}:")
    print(f"  Input: MedInc=${sample['MedInc']}k, Location={sample['ocean_proximity']}")
    print(f"  Prediction: ${result['predictions'][0] * 100000:,.0f}")
    print(f"  Latency: {result['latency_ms']:.1f}ms")

# Batch test
batch_result = simulate_api_predict(test_samples)
print(f"\nBatch test (n={len(test_samples)}):")
print(f"  Predictions: {[f'${p*100000:,.0f}' for p in batch_result['predictions']]}")
print(f"  Average latency per item: {batch_result['latency_ms']/len(test_samples):.1f}ms")

Simulated API Testing:

Test 1:
  Input: MedInc=$8.3252k, Location=NEAR_BAY
  Prediction: $423,406
  Latency: 159.5ms

Test 2:
  Input: MedInc=$3.5k, Location=INLAND
  Prediction: $197,327
  Latency: 150.2ms

Batch test (n=2):
  Predictions: ['$423,406', '$197,327']
  Average latency per item: 116.7ms


In [20]:
# Verify consistency between notebook and "API"
# Load test set and compare predictions
notebook_preds = loaded_model.predict(X_test)
api_preds = [simulate_api_predict(X_test.iloc[i].to_dict())['predictions'][0] 
             for i in range(len(X_test))]

# Check consistency
max_diff = np.max(np.abs(notebook_preds - api_preds))
print(f"Consistency Check:")
print(f"  Max difference between notebook and API: {max_diff:.10f}")
print(f"  Status: {'✓ PASS' if max_diff < 1e-10 else '✗ FAIL'}")

# Performance benchmark
import time
n_requests = 100
start = time.time()
for i in range(n_requests):
    _ = simulate_api_predict(X_test.iloc[i % len(X_test)].to_dict())
total_time = time.time() - start

print(f"\nPerformance Benchmark:")
print(f"  {n_requests} requests in {total_time:.2f}s")
print(f"  Throughput: {n_requests/total_time:.1f} req/sec")
print(f"  Average latency: {total_time/n_requests*1000:.1f}ms")

# Latency distribution
latencies = []
for i in range(50):
    start = time.time()
    simulate_api_predict(X_test.iloc[0].to_dict())
    latencies.append((time.time() - start) * 1000)

print(f"  Latency distribution (50 samples):")
print(f"    P50: {np.percentile(latencies, 50):.1f}ms")
print(f"    P95: {np.percentile(latencies, 95):.1f}ms")
print(f"    P99: {np.percentile(latencies, 99):.1f}ms")

Consistency Check:
  Max difference between notebook and API: 0.0000000000
  Status: ✓ PASS

Performance Benchmark:
  100 requests in 6.62s
  Throughput: 15.1 req/sec
  Average latency: 66.2ms
  Latency distribution (50 samples):
    P50: 65.7ms
    P95: 78.4ms
    P99: 81.2ms


## 🐳 7. Production Considerations & Next Steps

Moving from local development to production requires containerization, scaling strategies, and monitoring.

In [21]:
# Dockerfile template for FastAPI deployment
dockerfile_content = '''
# Dockerfile - Containerize the FastAPI app
FROM python:3.9-slim

# Set working directory
WORKDIR /app

# Copy requirements and install dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy model and application code
COPY housing_model_v1.joblib .
COPY fastapi_app.py .

# Expose port
EXPOSE 8000

# Run with uvicorn (production-grade server)
CMD ["uvicorn", "fastapi_app:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "4"]
'''

# requirements.txt
requirements_content = '''
fastapi==0.104.1
uvicorn[standard]==0.24.0
pydantic==2.5.0
scikit-learn==1.3.0
pandas==2.0.3
numpy==1.24.3
joblib==1.3.2
'''

print("Dockerfile for production deployment:")
print(dockerfile_content)

print("\nrequirements.txt:")
print(requirements_content)

print("\nBuild and run commands:")
print("  docker build -t housing-api .")
print("  docker run -p 8000:8000 housing-api")
print("  docker push registry.hub.docker.com/username/housing-api:latest")

Dockerfile for production deployment:

# Dockerfile - Containerize the FastAPI app
FROM python:3.9-slim

# Set working directory
WORKDIR /app

# Copy requirements and install dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy model and application code
COPY housing_model_v1.joblib .
COPY fastapi_app.py .

# Expose port
EXPOSE 8000

# Run with uvicorn (production-grade server)
CMD ["uvicorn", "fastapi_app:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "4"]


requirements.txt:

fastapi==0.104.1
uvicorn[standard]==0.24.0
pydantic==2.5.0
scikit-learn==1.3.0
pandas==2.0.3
numpy==1.24.3
joblib==1.3.2


Build and run commands:
  docker build -t housing-api .
  docker run -p 8000:8000 housing-api
  docker push registry.hub.docker.com/username/housing-api:latest


In [22]:
# Production scaling and monitoring concepts
production_guide = """
Production Deployment Checklist:

1. Environment Variables
   - MODEL_PATH: Path to model file (allows easy swapping)
   - LOG_LEVEL: INFO for prod, DEBUG for dev
   - PORT: 8000 default, configurable for cloud platforms
   - WORKERS: Number of uvicorn workers (typically 2-4 per CPU)

2. Scaling Strategies
   - Horizontal: Run multiple containers behind load balancer
   - Vertical: Increase CPU/memory per container
   - Auto-scaling: Kubernetes HPA based on CPU/request count

3. Monitoring & Observability
   - Prometheus metrics: request_count, latency_histogram, error_rate
   - Health checks: /health endpoint for load balancer
   - Distributed tracing: Track requests across services
   - Alerting: PagerDuty/Slack for error rate > 1%

4. Model Versioning & A/B Testing
   - Version in path: /v1/predict, /v2/predict
   - Traffic splitting: 90% v1, 10% v2 for canary
   - Feature flags: Enable new model per user segment

5. Security
   - HTTPS/TLS termination at load balancer
   - API key authentication for sensitive models
   - Input sanitization to prevent injection
   - Rate limiting: 100 req/min per IP

6. Cloud Deployment Options
   - AWS: ECS/Fargate or EKS for Kubernetes
   - GCP: Cloud Run (serverless) or GKE
   - Azure: Container Instances or AKS
   - Heroku: Simple git-push deployment
"""

print(production_guide)

# Environment configuration example
env_config = """
# .env file for local development
MODEL_PATH=./housing_model_v1.joblib
LOG_LEVEL=DEBUG
PORT=8000
WORKERS=1

# Production environment
MODEL_PATH=/models/housing_model_v1.joblib
LOG_LEVEL=INFO
PORT=8000
WORKERS=4
METRICS_PORT=9090
"""
print(env_config)


Production Deployment Checklist:

1. Environment Variables
   - MODEL_PATH: Path to model file (allows easy swapping)
   - LOG_LEVEL: INFO for prod, DEBUG for dev
   - PORT: 8000 default, configurable for cloud platforms
   - WORKERS: Number of uvicorn workers (typically 2-4 per CPU)

2. Scaling Strategies
   - Horizontal: Run multiple containers behind load balancer
   - Vertical: Increase CPU/memory per container
   - Auto-scaling: Kubernetes HPA based on CPU/request count

3. Monitoring & Observability
   - Prometheus metrics: request_count, latency_histogram, error_rate
   - Health checks: /health endpoint for load balancer
   - Distributed tracing: Track requests across services
   - Alerting: PagerDuty/Slack for error rate > 1%

4. Model Versioning & A/B Testing
   - Version in path: /v1/predict, /v2/predict
   - Traffic splitting: 90% v1, 10% v2 for canary
   - Feature flags: Enable new model per user segment

5. Security
   - HTTPS/TLS termination at load balancer
   - API key

## ⚠️ Common Pitfalls & Pro Tips

- **Loading model per request → slow**: Load model at startup (global scope) to avoid 100ms+ latency on every prediction
- **Not validating input types/shapes → crashes**: Use Pydantic or manual validation; never trust client input
- **Exposing raw predict without bounds → security risk**: Add rate limiting, authentication, and input sanitization for public APIs
- **No HTTPS in prod**: Always terminate TLS at load balancer or use HTTPS redirect; never transmit data unencrypted
- **Not handling missing features → KeyError**: Validate all required fields present before prediction; provide clear error messages
- **Forgetting to inverse_transform if scaled**: Ensure predictions are in correct units (our model outputs $100k directly, but some need scaling)
- **Large models → memory issues**: Use model compression (quantization, pruning) or load-on-demand for multi-model services
- **Not testing edge cases**: Test with missing values, extreme inputs, and malformed JSON before deployment
- **Synchronous only → poor throughput**: Use async endpoints (FastAPI) or worker pools (Gunicorn) for concurrent requests
- **No health checks**: Implement `/health` endpoint for load balancers to detect unhealthy instances
- **Logging to stdout only**: Persist logs to file or send to aggregation service (ELK, Splunk, Datadog)
- **Ignoring cold start**: Container startup time affects auto-scaling; keep images small and optimize imports

## 📝 Exercises

Practice deploying ML models:

**Easy:** Save the Iris classification pipeline from previous notebooks, then create a Flask `/predict` endpoint that accepts sepal/petal measurements and returns the predicted species.

**Medium:** Build a Streamlit app for California Housing with interactive sliders for all numeric features and a dropdown for ocean proximity. Add a visualization showing how changing median income affects the prediction.

**Medium:** Add Pydantic model validation to your FastAPI app. Include validators that check for reasonable ranges (e.g., house age < 200 years) and test with intentionally bad input to verify error handling.

**Hard:** Implement a batch prediction endpoint that accepts a list of houses and returns a list of predictions. Optimize for throughput by processing as a batch rather than looping through single predictions.

**Bonus:** Add basic logging to a file in your Flask app. Write a script that sends 100 requests with random data, then analyze the log file to find average latency and any error patterns.

<details>
<summary><b>Exercise Solutions (Click to Expand)</b></summary>

### Easy Solution Outline (Iris Flask API)
```python
# iris_app.py
from flask import Flask, request, jsonify
import joblib
import numpy as np

app = Flask(__name__)
model = joblib.load('iris_pipeline.joblib')

@app.route('/predict', methods=['POST'])
def predict():
    data = request.get_json()
    features = np.array([[data['sepal_length'], data['sepal_width'], 
                          data['petal_length'], data['petal_width']]])
    prediction = model.predict(features)[0]
    return jsonify({'species': int(prediction), 'species_name': ['setosa', 'versicolor', 'virginica'][prediction]})

if __name__ == '__main__':
    app.run(debug=True)
```

### Medium Solution Outline (Streamlit with Visualization)
```python
import streamlit as st
import plotly.express as px

# Create range of incomes
incomes = np.linspace(0.5, 15, 50)
predictions = []
base_input = {...}  # Base features
for inc in incomes:
    base_input['MedInc'] = inc
    predictions.append(model.predict(pd.DataFrame([base_input]))[0])

fig = px.line(x=incomes, y=predictions, labels={'x': 'Income', 'y': 'Price'})
st.plotly_chart(fig)
```

### Hard Solution Outline (Batch Prediction)
```python
@app.post("/predict_batch")
async def predict_batch(houses: List[HouseFeatures]):
    # Convert all to DataFrame at once (vectorized)
    df = pd.DataFrame([h.dict() for h in houses])
    predictions = model.predict(df)  # Single batch call, not loop
    return {'predictions': predictions.tolist(), 'count': len(houses)}
```

### Bonus Solution Outline (Logging Analysis)
```python
import re
from collections import Counter

# Parse log file
with open('api.log', 'r') as f:
    lines = f.readlines()

# Extract latencies
latencies = [float(re.search(r'latency_ms": (\d+)', line).group(1)) 
             for line in lines if 'latency_ms' in line]
print(f"Average latency: {np.mean(latencies):.1f}ms")
```

</details>

## ✅ Summary – What You Learned Today

- **Model serialization with joblib**: Save complete pipelines (preprocessing + model) for consistent deployment
- **Flask basics**: Build simple REST APIs with route decorators, request parsing, and JSON responses
- **FastAPI advantages**: Automatic validation via Pydantic, type-safe endpoints, async support, and auto-generated docs
- **Streamlit for UIs**: Create interactive web applications with pure Python — sliders, buttons, and visualizations
- **Input validation**: Use Pydantic models to enforce schemas and catch errors before they reach the model
- **Error handling**: Graceful failure modes with try/except, HTTP status codes, and meaningful error messages
- **Logging for observability**: Structured logging of predictions, latency, and errors for operational monitoring
- **Performance considerations**: Load models at startup, batch predictions when possible, measure latency percentiles
- **Production readiness**: Containerization with Docker, environment variables, health checks, and scaling strategies
- **Deployment patterns**: Stateless services enable horizontal scaling; separate model versioning from code versioning

